# 104 - Forbes Q Polynomials and Clenshaw Sums

The contemporary asphere made up of even radial powers has numerous well
documented traps: cancellation between orders, truncation of sig figs severely
changing the shape, and at times poor convergence in optimization.

Greg Forbes created a new family of three _slope_ orthogonal polynomials
that intend to correct this.  They are the Qbfs, Qcon, and "Q2D" sets.  Their
mathematical implementation is _very substantially_ more complex than any other
basis in prysm.  It is likely the case that a Zernike surface using only
radial modes is a "better" choice if you want to work with rotationally
symmetric surfaces, or a Zernike surface (or even XY) instead of Q2D for 2D.
Nevertheless, they are fully supported in prysm.

Because their implementation is primarily aimed at the raytracing module,
all three families also support Clenshaw's one-pass trick to compute the
surface sag and its spatial derivatives without ever actually computing
the bases.

First we'll show the three families:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from prysm.coordinates import make_xy_grid, cart_to_polar
from prysm.geometry import circle

# a 256x256 grid spanning the unit disk; r,t are polar; mask is True OUTSIDE
x, y = make_xy_grid(256, diameter=2)
r, t = cart_to_polar(x, y)
out = ~circle(1, r)        # boolean: True outside the unit circle (for blanking)

def show(arr, ax=None, blank=True, **kw):
    a = np.asarray(arr, dtype=float).copy()
    if blank:
        a[out] = np.nan
    ax = ax or plt.gca()
    return ax.imshow(a, cmap=kw.pop('cmap', 'RdBu'), **kw)

`Qbfs(n, r)` and `Qcon(n, r)` are rotationally symmetric. `Q2d(n, m, r, t)` is
made to be interface compatible to the Zernike implementation, and swaps cosine
or sine based on sign of m:

In [ ]:
from prysm.polynomials import Qbfs, Qcon, Q2d

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
show(Qbfs(2, r), axs[0]); axs[0].set(title='Qbfs(2)', xticks=[], yticks=[])
show(Qcon(2, r), axs[1]); axs[1].set(title='Qcon(2)', xticks=[], yticks=[])
show(Q2d(2, 2, r, t), axs[2]); axs[2].set(title='Q2d(2, 2)', xticks=[], yticks=[])

Sums work just like the other polynomials:

In [ ]:
from prysm.polynomials import Q2d_seq, sum_of_2d_modes

nms = [(0, 0), (1, 0), (2, 0), (1, 1), (2, -2), (3, 1)]
qbasis = Q2d_seq(nms, r, t)
coefs = np.array([0.0, 0.4, -0.15, 0.25, 0.2, -0.1])

sag = sum_of_2d_modes(qbasis, coefs)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = show(sag, ax); fig.colorbar(im, ax=ax, fraction=0.046)
ax.set(title='a freeform sag = sum of Q2d modes', xticks=[], yticks=[])

## The Clenshaw recurrence

Orthogonal polynomials obey a three-term recurrence, and Clenshaw's recurrence
uses it to evaluate a weighted sum directly - in fewer operations than forming
each term, and staying accurate at high order where the naive sum loses digits to
cancellation.  This is why the `_seq`/`_sum` routines exist instead of evaluating
each mode and adding.

`jacobi_sum_clenshaw(cs, alpha, beta, x)` exposes it for the Jacobi family.  We
check it against the naive term-by-term sum.

In [ ]:
from prysm.polynomials import jacobi_seq, jacobi_sum_clenshaw

x1 = x[0, :]
cs = np.array([0.5, -0.3, 0.8, 0.1, -0.2, 0.05])     # coefficients for orders 0..5

# naive: evaluate each order, scale, add
terms = jacobi_seq(range(len(cs)), 0, 0, x1)
naive = sum_of_2d_modes(terms, cs)

# Clenshaw: sum directly via the recurrence
clen = jacobi_sum_clenshaw(cs, 0, 0, x1)

print('max |Clenshaw - naive|:', np.nanmax(np.abs(clen - naive)))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x1, clen)
ax.set(title='a Jacobi series summed by Clenshaw recurrence', xlabel='x')

## Wrap-up

This notebook showed the Q polynomials, as well as using Clenshaw's technique to compute weighted sums.  The final notebook (201) will walk through spatial derivatives of any of the bases.